---
title: 'Solutions: Measuring Distribution Distances'
author: Mark Fuge
format:
    html:
        code-fold: true
        code-summary: "Show Solution Code"
---

Worked solutions to the exercises at the end of the chapter *Measuring Distribution Distances*. The first two code cells set up the chapter's distributions and distance functions; each solution then reuses them.

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Callable, Dict, Tuple

import numpy as np
import seaborn as sns
import torch
from matplotlib import pyplot as plt
from matplotlib.gridspec import GridSpec

sns.set_context("talk")
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
DEFAULT_DTYPE = torch.float64
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True})

In [2]:
# The chapter's distributions and distance functions, exactly as the chapter defines them
GRID_X = torch.linspace(-20.0, 20.0, 8192, dtype=DEFAULT_DTYPE, device=device)


DX = GRID_X[1] - GRID_X[0]


EPS = torch.tensor(1e-12, dtype=DEFAULT_DTYPE, device=device)


PLOT_XLIM = (-6.0, 6.0)


def gaussian_pdf_torch(x: torch.Tensor, mean: torch.Tensor | float, std: torch.Tensor | float) -> torch.Tensor:
    """Evaluate the Gaussian pdf on `x` using broadcasted parameters."""
    mean_t = torch.as_tensor(mean, dtype=DEFAULT_DTYPE, device=device)
    std_t = torch.as_tensor(std, dtype=DEFAULT_DTYPE, device=device)
    var = std_t ** 2
    norm_const = 1.0 / torch.sqrt(2.0 * math.pi * var)
    z = (x - mean_t) ** 2 / (2.0 * var)
    return norm_const * torch.exp(-z)


def gaussian_log_pdf_torch(x: torch.Tensor, mean: torch.Tensor | float, std: torch.Tensor | float) -> torch.Tensor:
    """Log of the Gaussian pdf, evaluated directly so that far tails do not underflow to zero."""
    mean_t = torch.as_tensor(mean, dtype=DEFAULT_DTYPE, device=device)
    std_t = torch.as_tensor(std, dtype=DEFAULT_DTYPE, device=device)
    return -0.5 * ((x - mean_t) / std_t) ** 2 - torch.log(std_t) - 0.5 * math.log(2.0 * math.pi)


def uniform_pdf_torch(x: torch.Tensor, low: float, high: float) -> torch.Tensor:
    low_t = torch.as_tensor(low, dtype=DEFAULT_DTYPE, device=device)
    high_t = torch.as_tensor(high, dtype=DEFAULT_DTYPE, device=device)
    inside = (x >= low_t) & (x <= high_t)
    return inside.to(DEFAULT_DTYPE) / (high_t - low_t + EPS)


def gaussian_mixture_pdf_torch(
    x: torch.Tensor,
    means: torch.Tensor,
    stds: torch.Tensor,
    weights: torch.Tensor,
) -> torch.Tensor:
    comps = [w * gaussian_pdf_torch(x, m, s) for w, m, s in zip(weights, means, stds)]
    return torch.stack(comps, dim=0).sum(dim=0)


@dataclass(frozen=True)
class DistributionSpec:
    name: str
    pdf: Callable[[torch.Tensor], torch.Tensor]
    sampler: Callable[[int, torch.Tensor | None], torch.Tensor]
    metadata: Dict[str, float]


def make_gaussian(mean: float = 0.0, std: float = 1.0) -> DistributionSpec:
    mean_t = torch.tensor(mean, dtype=DEFAULT_DTYPE, device=device)
    std_t = torch.tensor(std, dtype=DEFAULT_DTYPE, device=device)

    def _pdf(x: torch.Tensor) -> torch.Tensor:
        return gaussian_pdf_torch(x, mean_t, std_t)

    def _sample(n: int, base_noise: torch.Tensor | None = None) -> torch.Tensor:
        eps = torch.randn(n, dtype=DEFAULT_DTYPE, device=device) if base_noise is None else base_noise
        return mean_t + std_t * eps

    meta = {"mean": float(mean), "std": float(std)}
    return DistributionSpec(name=f"Gaussian(μ={mean:.2f}, σ={std:.2f})", pdf=_pdf, sampler=_sample, metadata=meta)


def make_uniform(low: float = -2.0, high: float = 2.0) -> DistributionSpec:
    low_t = torch.tensor(low, dtype=DEFAULT_DTYPE, device=device)
    high_t = torch.tensor(high, dtype=DEFAULT_DTYPE, device=device)

    def _pdf(x: torch.Tensor) -> torch.Tensor:
        return uniform_pdf_torch(x, float(low_t), float(high_t))

    def _sample(n: int, base_noise: torch.Tensor | None = None) -> torch.Tensor:
        if base_noise is None:
            base_noise = torch.rand(n, dtype=DEFAULT_DTYPE, device=device)
        return low_t + (high_t - low_t) * base_noise

    meta = {"low": float(low), "high": float(high)}
    return DistributionSpec(name=f"Uniform[{low:.1f}, {high:.1f}]", pdf=_pdf, sampler=_sample, metadata=meta)


def make_gaussian_mixture(
    means: Tuple[float, float] = (-2.0, 2.0),
    stds: Tuple[float, float] = (0.6, 0.6),
    weights: Tuple[float, float] = (0.5, 0.5),
) -> DistributionSpec:
    means_t = torch.tensor(means, dtype=DEFAULT_DTYPE, device=device)
    stds_t = torch.tensor(stds, dtype=DEFAULT_DTYPE, device=device)
    weights_t = torch.tensor(weights, dtype=DEFAULT_DTYPE, device=device)
    weights_t = weights_t / weights_t.sum()

    def _pdf(x: torch.Tensor) -> torch.Tensor:
        return gaussian_mixture_pdf_torch(x, means_t, stds_t, weights_t)

    def _sample(n: int, base_noise: torch.Tensor | None = None) -> torch.Tensor:
        # `base_noise` is standard-normal noise shared with the model (see sample_with_matching_noise);
        # the component of each sample is assigned deterministically in the mixture proportions.
        eps = torch.randn(n, dtype=DEFAULT_DTYPE, device=device) if base_noise is None else base_noise
        cumulative = torch.cumsum(weights_t, dim=0)
        u = torch.linspace(0.0, 1.0, n, dtype=DEFAULT_DTYPE, device=device)
        choices = torch.clamp(torch.bucketize(u, cumulative), max=len(means_t) - 1)
        return means_t[choices] + stds_t[choices] * eps

    meta = {
        "means": tuple(float(m) for m in means),
        "stds": tuple(float(s) for s in stds),
        "weights": tuple(float(w) for w in weights),
    }
    desc = ", ".join([f"μ={m:.1f}" for m in means])
    return DistributionSpec(name=f"Gaussian Mixture ({desc})", pdf=_pdf, sampler=_sample, metadata=meta)


def plot_distribution_pair(
    grid: torch.Tensor,
    target_pdf: torch.Tensor,
    model_pdf: torch.Tensor,
    title: str,
    target_label: str = "Target",
    model_label: str = "Model",
) -> None:
    fig, ax = plt.subplots()
    ax.plot(grid.cpu().numpy(), target_pdf.cpu().numpy(), label=target_label, linewidth=2)
    ax.plot(grid.cpu().numpy(), model_pdf.cpu().numpy(), label=model_label, linestyle="--", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("Density")
    ax.set_xlim(*PLOT_XLIM)
    ax.legend()
    plt.show()


def evaluate_metric_curve(
    mu_values: np.ndarray,
    metric_fn: Callable[[torch.Tensor], torch.Tensor],
) -> Tuple[np.ndarray, np.ndarray]:
    metrics = []
    grads = []
    for mu in mu_values:
        mu_t = torch.tensor(mu, dtype=DEFAULT_DTYPE, device=device, requires_grad=True)
        metric = metric_fn(mu_t)
        grad = torch.autograd.grad(metric, mu_t)[0]
        metrics.append(metric.detach().cpu().item())
        grads.append(grad.detach().cpu().item())
    return np.array(metrics), np.array(grads)


TARGETS: Dict[str, DistributionSpec] = {
    "gaussian": make_gaussian(mean=0.0, std=1.0),
    "wide_gaussian": make_gaussian(mean=0.0, std=1.8),
    "uniform": make_uniform(low=-3.0, high=3.0),
    "gaussian_mixture": make_gaussian_mixture(means=(-2.5, 2.5), stds=(0.6, 0.6), weights=(0.35, 0.65)),
}


_target_pdf_cache: Dict[str, torch.Tensor] = {}


def get_target_pdf(spec: DistributionSpec) -> torch.Tensor:
    if spec.name not in _target_pdf_cache:
        with torch.no_grad():
            _target_pdf_cache[spec.name] = spec.pdf(GRID_X).detach()
    return _target_pdf_cache[spec.name]


def forward_kl(mu: torch.Tensor, target: DistributionSpec, model_std: float = 1.0) -> torch.Tensor:
    p_pdf = get_target_pdf(target)
    log_q = gaussian_log_pdf_torch(GRID_X, mu, model_std)
    integrand = p_pdf * (torch.log(p_pdf + EPS) - log_q)
    return torch.trapz(integrand, GRID_X)


def reverse_kl(mu: torch.Tensor, target: DistributionSpec, model_std: float = 1.0) -> torch.Tensor:
    p_pdf = get_target_pdf(target)
    log_q = gaussian_log_pdf_torch(GRID_X, mu, model_std)
    q_pdf = torch.exp(log_q)
    integrand = q_pdf * (log_q - torch.log(p_pdf + EPS))
    return torch.trapz(integrand, GRID_X)


def jensen_shannon(mu: torch.Tensor, target: DistributionSpec, model_std: float = 1.0) -> torch.Tensor:
    p_pdf = get_target_pdf(target)
    q_pdf = gaussian_pdf_torch(GRID_X, mu, model_std)
    m_pdf = 0.5 * (p_pdf + q_pdf)
    term_p = p_pdf * torch.log((p_pdf + EPS) / (m_pdf + EPS))
    term_q = q_pdf * torch.log((q_pdf + EPS) / (m_pdf + EPS))
    return 0.5 * torch.trapz(term_p, GRID_X) + 0.5 * torch.trapz(term_q, GRID_X)


BASE_NOISE_NORMAL = torch.randn(4096, dtype=DEFAULT_DTYPE, device=device)


def sample_with_matching_noise(spec: DistributionSpec, sample_count: int) -> torch.Tensor:
    if "Uniform" in spec.name:
        # Evenly spaced levels across all of [0, 1], so the samples cover the whole uniform support
        base = (torch.arange(sample_count, dtype=DEFAULT_DTYPE, device=device) + 0.5) / sample_count
    else:
        base = BASE_NOISE_NORMAL[:sample_count]
    return spec.sampler(sample_count, base)


def rbf_kernel(x: torch.Tensor, y: torch.Tensor, bandwidth: float) -> torch.Tensor:
    bw = torch.as_tensor(bandwidth, dtype=DEFAULT_DTYPE, device=device)
    diff = x[:, None] - y[None, :]
    return torch.exp(-0.5 * diff.pow(2) / (bw ** 2 + EPS))


def maximum_mean_discrepancy(
    mu: torch.Tensor,
    target: DistributionSpec,
    model_std: float = 1.0,
    bandwidth: float = 1.0,
    sample_count: int = 512,
) -> torch.Tensor:
    samples_p = sample_with_matching_noise(target, sample_count)
    base_gauss = BASE_NOISE_NORMAL[:sample_count]
    samples_q = mu + model_std * base_gauss
    xx = rbf_kernel(samples_p, samples_p, bandwidth)
    yy = rbf_kernel(samples_q, samples_q, bandwidth)
    xy = rbf_kernel(samples_p, samples_q, bandwidth)
    xx = xx - torch.diag(torch.diag(xx))
    yy = yy - torch.diag(torch.diag(yy))
    n = sample_count
    mmd2 = xx.sum() / (n * (n - 1) + EPS) - 2.0 * xy.mean() + yy.sum() / (n * (n - 1) + EPS)
    return torch.sqrt(torch.clamp(mmd2, min=0.0))


def gaussian_quantile(u: torch.Tensor, mean: float, std: float) -> torch.Tensor:
    mean_t = torch.as_tensor(mean, dtype=DEFAULT_DTYPE, device=device)
    std_t = torch.as_tensor(std, dtype=DEFAULT_DTYPE, device=device)
    return mean_t + std_t * math.sqrt(2.0) * torch.erfinv(2.0 * u - 1.0)


def wasserstein_2(
    mu: torch.Tensor,
    target: DistributionSpec,
    model_std: float = 1.0,
    sample_count: int = 1024,
) -> torch.Tensor:
    if "mean" in target.metadata and "std" in target.metadata:  # single Gaussian target: use closed form
        mu_p = target.metadata.get("mean", 0.0)
        sigma_p = target.metadata.get("std", 1.0)
        sigma_q = model_std
        diff_sq = (mu - mu_p) ** 2 + (sigma_q - sigma_p) ** 2
        return torch.sqrt(torch.clamp(diff_sq, min=0.0))

    # Empirical quantile-based estimate for non-Gaussian targets
    u = torch.linspace(0.0 + 0.5 / sample_count, 1.0 - 0.5 / sample_count, sample_count, dtype=DEFAULT_DTYPE, device=device)
    samples_p = torch.sort(sample_with_matching_noise(target, sample_count))[0]
    samples_q = torch.sort(mu + model_std * BASE_NOISE_NORMAL[:sample_count])[0]
    return torch.sqrt(torch.mean((samples_p - samples_q) ** 2))


### Exercise 1: Five distances, four properties

**(a)**

| Distance | Symmetric | Bounded | Finite for non-overlapping supports | True metric |
|---|---|---|---|---|
| Forward KL $\mathrm{KL}(p\,\parallel\,q)$ | no: the expectation is taken under $p$, and swapping the roles changes the value | no: $\mathrm{KL}(\mathcal{N}(0,1)\,\parallel\,\mathcal{N}(\mu,1)) = \mu^2/2$ grows without limit | no: wherever $p > 0$ and $q = 0$ the integrand $p \log(p/0)$ is infinite | no: not symmetric and no triangle inequality |
| Reverse KL $\mathrm{KL}(q\,\parallel\,p)$ | no: same reason | no: same reason | no: wherever $q > 0$ and $p = 0$ it is infinite | no |
| Jensen-Shannon | yes: $p$ and $q$ enter through the symmetric mixture $m$ | yes: at most $\log 2$ | yes: $m \ge p/2$ wherever $p > 0$, so no log ratio exceeds $\log 2$ | no as it stands, although $\sqrt{\mathrm{JS}}$ is a metric |
| MMD (RBF kernel) | yes: the formula is symmetric in the two sample sets | yes: with $0 < k \le 1$, $\mathrm{MMD}^2 \le 2$ | yes: it only ever evaluates kernel values | yes, for a characteristic kernel such as the RBF kernel |
| Wasserstein-2 | yes: the transport cost from $p$ to $q$ equals that from $q$ to $p$ | no: it grows like the distance the mass has to travel | yes: that distance is exactly what it measures | yes |

**(b)** The cell below checks four entries: the two KL directions differ, the JS divergence is the same in both directions, the largest JS value over a wide sweep equals $\log 2$, and $W_2$ keeps growing. A numerical check only evaluates the distance for *particular* pairs of distributions. One pair for which $\mathrm{KL}(p\,\parallel\,q) \ne \mathrm{KL}(q\,\parallel\,p)$ can show that symmetry doesn't hold, and providing one pair with a value above a proposed bound refutes boundedness, but no finite number of pairs can show that *every* pair behaves -- for this we would need a formal argument or proof.

In [3]:
p_standard = TARGETS["gaussian"]
q_shifted = make_gaussian(mean=1.5, std=0.5)
mu_q = torch.tensor(1.5, dtype=DEFAULT_DTYPE, device=device)
mu_p = torch.tensor(0.0, dtype=DEFAULT_DTYPE, device=device)

kl_pq = forward_kl(mu_q, p_standard, model_std=0.5).item()           # KL(N(0,1) || N(1.5, 0.5^2))
kl_qp = reverse_kl(mu_q, p_standard, model_std=0.5).item()           # KL(N(1.5, 0.5^2) || N(0,1))
js_one_way = jensen_shannon(mu_q, p_standard, model_std=0.5).item()  # JS(N(0,1), N(1.5, 0.5^2))
js_other_way = jensen_shannon(mu_p, q_shifted, model_std=1.0).item()  # JS(N(1.5, 0.5^2), N(0,1))
print(f"KL(p||q) = {kl_pq:.4f}   KL(q||p) = {kl_qp:.4f}   -> not symmetric")
print(f"JS(p, q) = {js_one_way:.4f}   JS(q, p) = {js_other_way:.4f}   -> symmetric")

mu_sweep = np.linspace(-15.0, 15.0, 121)
js_sweep, _ = evaluate_metric_curve(mu_sweep, lambda mu: jensen_shannon(mu, p_standard))
w2_sweep, _ = evaluate_metric_curve(mu_sweep, lambda mu: wasserstein_2(mu, p_standard))
mmd_sweep, _ = evaluate_metric_curve(mu_sweep, lambda mu: maximum_mean_discrepancy(mu, p_standard, bandwidth=1.0))
print(f"largest JS over the sweep = {js_sweep.max():.4f}   (log 2 = {math.log(2):.4f})")
print(f"largest MMD over the sweep = {mmd_sweep.max():.4f}   (sqrt 2 = {math.sqrt(2):.4f})")
print(f"W2 at mu = 15: {w2_sweep[-1]:.1f}, at mu = 7.5: {w2_sweep[90]:.1f}   -> keeps growing")

KL(p||q) = 5.3069   KL(q||p) = 1.4431   -> not symmetric
JS(p, q) = 0.3512   JS(q, p) = 0.3512   -> symmetric
largest JS over the sweep = 0.6931   (log 2 = 0.6931)
largest MMD over the sweep = 1.0852   (sqrt 2 = 1.4142)
W2 at mu = 15: 15.0, at mu = 7.5: 7.5   -> keeps growing


### Exercise 2: Which fit has the lowest reverse KL?

In [4]:
EX2_TARGET = TARGETS["gaussian_mixture"]
EX2_CANDIDATES = {                 # letter: (mu, sigma), as in the chapter's figure
    "A": (-2.5, 1.5),
    "B": (0.0, 1.0),
    "C": (2.5, 0.6),
    "D": (0.75, 2.45),
    "E": (2.5, 0.3),
    "F": (-2.5, 0.6),
}
EX2_DESCRIPTION = {
    "A": "flat fit on the smaller mode",
    "B": "sitting between the modes",
    "C": "mode-width fit on the taller mode",
    "D": "broad fit covering both modes",
    "E": "narrower than the taller mode",
    "F": "mode-width fit on the smaller mode",
}

ex2_rows = []
for letter, (mu_c, sigma_c) in EX2_CANDIDATES.items():
    mu_t = torch.tensor(mu_c, dtype=DEFAULT_DTYPE, device=device)
    ex2_rows.append((letter, forward_kl(mu_t, EX2_TARGET, model_std=sigma_c).item(),
                     reverse_kl(mu_t, EX2_TARGET, model_std=sigma_c).item()))
print("candidate   KL(p||q)   KL(q||p)   description")
for letter, fkl, rkl in sorted(ex2_rows, key=lambda row: row[2]):
    print(f"    {letter}      {fkl:8.3f}   {rkl:8.3f}   {EX2_DESCRIPTION[letter]}")
ex2_reverse = {letter: rkl for letter, _, rkl in ex2_rows}
print(f"\nF minus C = {ex2_reverse['F'] - ex2_reverse['C']:.3f}   log(0.65 / 0.35) = {math.log(0.65 / 0.35):.3f}")
print(f"E minus C = {ex2_reverse['E'] - ex2_reverse['C']:.3f}   "
      f"KL(N(2.5, 0.3^2) || N(2.5, 0.6^2)) = {math.log(0.6 / 0.3) + 0.3**2 / (2 * 0.6**2) - 0.5:.3f}")

candidate   KL(p||q)   KL(q||p)   description
    C        11.505      0.431   mode-width fit on the taller mode
    E        48.771      0.749   narrower than the taller mode
    F        21.922      1.050   mode-width fit on the smaller mode
    D         0.763      2.252   broad fit covering both modes
    A         3.460      2.295   flat fit on the smaller mode
    B         2.668      4.206   sitting between the modes

F minus C = 0.619   log(0.65 / 0.35) = 0.619
E minus C = 0.318   KL(N(2.5, 0.3^2) || N(2.5, 0.6^2)) = 0.318


**(a)** The reverse KL is an expectation under $q$, so all that matters is what $p$ looks like where $q$ puts its mass, plus the entropy of $q$. Candidate C sits on the taller mode with exactly the mode's width, so $p(x) \approx 0.65\,q(x)$ everywhere $q$ lives, and $\mathrm{KL}(q\,\parallel\,p) \approx -\log 0.65 \approx 0.43$. E sits on the same mode but is narrower: it also sees only good values of $p$, but it pays in entropy. F is the mirror image of C on the smaller mode and pays $-\log 0.35$ instead of $-\log 0.65$. D and A spread mass into the valley around $x = 0$, where $p$ is tiny, and B sits entirely in that valley. The ranking is C, E, F, then D and A, and B last.

**(b)** The printed table confirms the order: C $0.43$, E $0.75$, F $1.05$, D $2.25$, A $2.30$, B $4.21$. The entry most people misjudge is D against A: the broad candidate covers both modes, which looks like a virtue, but under the reverse KL it is penalized for every bit of mass in the valley just as A is, and its low entropy advantage is small. Under the forward KL, by contrast, D wins by a wide margin ($0.76$ against $3.46$ for A) and C and E are terrible ($11.5$ and $48.8$), because the forward KL is an expectation under $p$ and punishes $q$ for being near zero on the mode it ignores.

**(c)** The two directions ask different questions. $\mathrm{KL}(p\,\parallel\,q) = \int p \log (p/q)$ samples $x$ from the target, so $q$ must be large wherever $p$ is: a broad $q$ never leaves a mode uncovered, and it pays only the modest price of being too low at the peaks. $\mathrm{KL}(q\,\parallel\,p) = \int q \log (q/p)$ samples $x$ from the model, so $q$ must not put mass where $p$ is small: a broad $q$ spreads a lot of mass over the valley between the modes, where $\log(q/p)$ is large, and that is what sinks it. The same density is "safe" under one direction and "reckless" under the other.

**(d)** C and F both match the shape of their component exactly, so inside each one $p(x) = w\,q(x)$ with $w$ the mixture weight, and $\mathrm{KL}(q\,\parallel\,p) = -\log w$. The difference is therefore $\log(0.65/0.35) \approx 0.62$, which the table reproduces. The size of the difference is set by the ratio of the mixture weights, not by the distance between the modes.

**(e)** It is the entropy term. Going from C to E halves $\sigma$, which raises $-H(q)$ by $\log 2 \approx 0.69$, while the expected log-density $\mathbb{E}_q[\log p]$ improves only by about $0.37$ because $q$ now sits closer to the peak of $p$. The net effect is $+0.32$, exactly $\mathrm{KL}(\mathcal{N}(2.5, 0.3^2)\,\parallel\,\mathcal{N}(2.5, 0.6^2))$. Being more confident than the data warrant is a cost under the reverse KL, not a gain.

**(f)** A model trained with forward KL behaves like D: it covers both design families, but it also produces designs that lie between them, which do not belong to either family and are usually not valid designs at all (blurry or averaged samples). A model trained with reverse KL behaves like C: every sample looks like a plausible design, but one family is simply missing. This is the mode-covering versus mode-seeking trade-off of the chapter, seen from the training objective's side.

### Exercise 3: Flat divergences, flat gradients

In [5]:
mu_sweep = np.linspace(-15.0, 15.0, 121)
fkl_sweep, fkl_grad = evaluate_metric_curve(mu_sweep, lambda mu: forward_kl(mu, TARGETS["gaussian"]))
js_sweep, js_grad = evaluate_metric_curve(mu_sweep, lambda mu: jensen_shannon(mu, TARGETS["gaussian"]))
print("  mu   forward KL   d(fKL)/dmu      JS      d(JS)/dmu")
for mu_value in [2.0, 4.0, 6.0, 8.0, 10.0, 15.0]:
    i = int(np.argmin(np.abs(mu_sweep - mu_value)))
    print(f"{mu_value:4.0f}   {fkl_sweep[i]:8.2f}   {fkl_grad[i]:9.3f}   {js_sweep[i]:.5f}   {js_grad[i]:10.2e}")
i10 = int(np.argmin(np.abs(mu_sweep - 10.0)))
print(f"\nlog 2 = {math.log(2):.5f};  gradient ratio at mu = 10: {fkl_grad[i10] / js_grad[i10]:.1e}")

  mu   forward KL   d(fKL)/dmu      JS      d(JS)/dmu
   2       2.00       2.000   0.33683     2.25e-01
   4       8.00       4.000   0.63272     6.86e-02
   6      18.00       6.000   0.68930     6.23e-03
   8      32.00       8.000   0.69305     1.96e-04
  10      50.00      10.000   0.69315     2.23e-06
  15     112.50      15.000   0.69315    -5.15e-07

log 2 = 0.69315;  gradient ratio at mu = 10: 4.5e+06


**(a)** Beyond $|\mu| \approx 6$ the JS divergence is flat at $\log 2 \approx 0.693$ (it is within $1$ percent of that value at $\mu = 6$ and indistinguishable from it at $\mu = 8$). The JS divergence compares each distribution with the mixture $m = \tfrac{1}{2}(p + q)$. Once $p$ and $q$ no longer overlap, $m$ equals $p/2$ wherever $p$ lives and $q/2$ wherever $q$ lives, so both KL terms equal $\log 2$ and nothing changes any more as the distributions move further apart: the JS divergence can only see *whether* the two overlap, not *how far* apart they are once they do not.

**(b)** On the logarithmic panel the forward KL gradient magnitude at $\mu = 10$ is $10$ (the closed form gives $\mathrm{KL} = \mu^2/2$, so the gradient is $\mu$), while the JS gradient magnitude has fallen to about $2 \times 10^{-6}$: the two differ by more than six orders of magnitude, and the JS curve on that panel keeps dropping as a straight line, which is what a Gaussian tail looks like on a log axis. The forward KL gradient grows linearly with the distance because it is an expectation under $p$ of $\log q$, and $\log q$ is a parabola in $\mu$ no matter how small $q$ is. The JS gradient is proportional to the overlap of the two densities, which decays like a Gaussian tail.

**(c)** Under the forward KL, the model starting at $\mu = 10$ receives a gradient of $10$ and the first steps move it quickly and in the right direction; the step shrinks as it approaches the data, and the whole trajectory is well behaved. Under the JS divergence the model sits on a plateau: the loss is $\log 2$ in every direction, the gradient is numerically zero, and gradient descent does not move. The model only starts learning once some chance overlap appears. This is the vanishing-gradient problem of a generative adversarial network whose discriminator is too good: the generator's loss is flat, and it is one of the reasons later chapters replace the JS divergence by distances such as $W_2$ that keep a slope.

### Exercise 4: Predict the MMD curve

**(a)** Candidate C. B cannot be right because the bandwidth changes every kernel value in the formula, so the MMD cannot be independent of it. D cannot be right because for a very large bandwidth every kernel value tends to $1$ and the three terms of $\mathrm{MMD}^2$ cancel, so the curve must come back down. A has the right shape but its peak sits near $\sigma_k \approx 20$, a kernel so wide that two Gaussians two units apart look identical to it; the MMD is largest when the kernel resolves differences on the scale at which $P$ and $Q$ actually differ. The real curve peaks at $\sigma_k \approx 1.1$, which is that scale: the means are $2$ apart and each distribution has standard deviation $1$. Much narrower kernels see only the noise between individual samples, and much wider kernels cannot see a two-unit shift at all.

**(b)** For $\sigma_k$ far below the typical spacing between samples, $k_\sigma(x, y) \approx 0$ for every pair $x \ne y$. The two within-sample terms (whose diagonals the chapter removes) and the cross term all tend to zero, so $\mathrm{MMD}^2 \to 0$ from the left. For $\sigma_k$ far above every distance in the data, $k_\sigma(x, y) \approx 1 - (x - y)^2 / (2\sigma_k^2)$ for every pair, so all three terms are close to $1$ and $\mathrm{MMD}^2 \approx 1 - 2 + 1 = 0$; what survives is the second-order term, which is proportional to $(\mu_P - \mu_Q)^2 / \sigma_k^2$, and that is why the right tail decays like $1/\sigma_k$ on the log axis.

**(c)** For any two distributions the MMD tends to zero as the bandwidth tends to infinity, because the kernel then assigns similarity one to every pair of points and cannot distinguish any two sample sets. That is not a flaw: the bandwidth *is* the resolution at which the kernel compares distributions, and a kernel that is blind at a given scale cannot report differences at that scale. In practice $\sigma_k$ is chosen from the data, for example as the median pairwise distance between samples, or several bandwidths are summed so that differences at several scales are all visible.

### Exercise 5: Supports that do not overlap

In [6]:
EX5_TARGET = make_uniform(low=-3.0, high=-1.0)
EX5_MU = 3.0
EX5_MODEL_STD = 1.0

ex5_distances = {
    "forward KL": lambda mu: forward_kl(mu, EX5_TARGET, model_std=EX5_MODEL_STD),
    "reverse KL": lambda mu: reverse_kl(mu, EX5_TARGET, model_std=EX5_MODEL_STD),
    "Jensen-Shannon": lambda mu: jensen_shannon(mu, EX5_TARGET, model_std=EX5_MODEL_STD),
    "MMD, bandwidth 0.3": lambda mu: maximum_mean_discrepancy(mu, EX5_TARGET, model_std=EX5_MODEL_STD, bandwidth=0.3),
    "MMD, bandwidth 1.0": lambda mu: maximum_mean_discrepancy(mu, EX5_TARGET, model_std=EX5_MODEL_STD, bandwidth=1.0),
    "MMD, bandwidth 2.5": lambda mu: maximum_mean_discrepancy(mu, EX5_TARGET, model_std=EX5_MODEL_STD, bandwidth=2.5),
}
ex5_results = {}
print(f"{'distance':20s}   value      d/dmu")
for name, distance in ex5_distances.items():
    mu_t = torch.tensor(EX5_MU, dtype=DEFAULT_DTYPE, device=device, requires_grad=True)
    value = distance(mu_t)
    gradient = torch.autograd.grad(value, mu_t)[0].item()
    ex5_results[name] = (value.item(), gradient)
    print(f"{name:20s}   {value.item():7.3f}   {gradient:9.5f}")
print(f"\nlog(1 / EPS) = {-math.log(EPS.item()):.1f};  the mean of the target is {-2.0:.1f}, so mu - E_p[x] = {EX5_MU + 2.0:.1f}")

distance               value      d/dmu
forward KL              12.873     4.99304
reverse KL              26.211     0.00359
Jensen-Shannon           0.693     0.00061
MMD, bandwidth 0.3       0.737     0.00000
MMD, bandwidth 1.0       1.161     0.00368
MMD, bandwidth 2.5       1.225     0.09069

log(1 / EPS) = 27.6;  the mean of the target is -2.0, so mu - E_p[x] = 5.0


**(a)** and **(b)** The table above gives the numbers. Most predictions about the *values* come out as expected: the forward and reverse KL are large (and the reverse KL would be infinite without the chapter's `EPS`), the JS divergence sits at its ceiling $\log 2$, and each MMD sits at its ceiling for its bandwidth. The gradients are where predictions tend to go wrong. The forward KL gradient is $5.0$: large, correct in sign, and exactly $\mu - \mathbb{E}_p[x]$, because for a Gaussian model $\mathbb{E}_p[-\log q]$ is a parabola in $\mu$ whatever the target looks like (this is the moment-matching property from the forward KL section). Everything else is flat: the reverse KL and JS gradients are of order $10^{-3}$ and $10^{-4}$, and the MMD gradient is zero for the narrowest kernel, about $0.004$ for bandwidth $1$ and about $0.09$ for the widest one.

**(c)** Ranked by usefulness of the gradient: forward KL first, because its gradient points the right way and grows with the distance, which makes it strong here (though it also makes the step size hard to choose); MMD with the widest bandwidth second, because its gradient is small but nonzero and would become useful once the model came within a few bandwidths of the data; and the reverse KL, the JS divergence and the two narrower MMDs last, because their gradients are zero or close to it. The values tell a different story because a value says *how different* two distributions are, while a gradient says *which way to move*: a distance can be at its maximum and still carry no directional information, which is exactly what a bounded or support-based distance does once the supports have separated.

**(d)** The reverse KL of about $26$ is "infinite in spirit": wherever $q > 0$ and $p = 0$ the integrand is $q \log(q / 0)$, and only the `EPS` added to $p$ inside the logarithm keeps it at $\log(1/\mathrm{EPS}) \approx 27.6$ per unit of model mass. Its near-zero gradient says that moving $\mu$ by a little changes nothing: all of the model's mass is still outside the support. The JS divergence at $\log 2$ with a zero gradient says the same thing in bounded form, and the MMD near its plateau says that with a kernel of bandwidth $0.3$ no sample of the model is within a few bandwidths of any sample of the target, so the kernel cannot even see which direction the data lie in; with bandwidth $1$ only the model's far left tail comes close enough, which is why that gradient is small but not zero. The forward KL is finite only because the Gaussian model has tails that reach the target, so $\log q$ is large and negative but not infinite; its value of about $13$ is dominated by $\mathbb{E}_p[(x - \mu)^2]/2$.